In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Singularity Machine Learning - Classification：Multiverse Computing による Qiskit Function
*[API リファレンス](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity)を参照してください*

<Accordion>
<AccordionItem title="パッケージのバージョン">

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降の使用を推奨します。

```
scikit-learn~=1.8.0
```
</AccordionItem>
</Accordion>
> **Note:** * Qiskit Functions は、IBM Quantum&reg; Premium Plan・Flex Plan・On-Prem（IBM Quantum Platform API 経由）プランのユーザーのみが利用できる実験的機能です。現在プレビュー リリースの段階にあり、変更される場合があります。

## 概要
「Singularity Machine Learning - Classification」関数を使用すると、量子の専門知識を必要とせずに、実世界の機械学習問題を量子ハードウェア上で解くことができます。アンサンブル手法をベースにしたこの Application 関数は、ハイブリッド分類器です。初期のアンサンブル学習にはブースティング・バギング・スタッキングといった古典的な手法を活用し、その後、変分量子固有値ソルバー（VQE）や量子近似最適化アルゴリズム（QAOA）などの量子アルゴリズムを用いて、学習済みアンサンブルの多様性・汎化能力・全体的な複雑さを向上させます。

他の量子機械学習ソリューションとは異なり、この関数は QPU の量子ビット数に制限されることなく、数百万のサンプルと特徴量を持つ大規模データセットを処理できます。量子ビット数は学習できるアンサンブルのサイズのみを決定します。また、高い柔軟性を持ち、金融・医療・サイバーセキュリティなど幅広い分野の分類問題に適用可能です。
高次元・ノイズが多い・クラス不均衡といった古典的に困難な問題においても、一貫して高い精度を達成します。
![動作原理](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
この関数は、以下のような方々のために構築されています。
1. 量子機械学習を自社の製品やサービスに統合することで技術力強化を図る企業のエンジニアおよびデータサイエンティスト、
2. 量子機械学習の応用を探求し、分類タスクに量子コンピューティングを活用しようとしている量子研究所の研究者、
3. 機械学習などの講義で量子コンピューティングの優位性を示したい教育機関の学生および教員。

以下の例では、`create`・`list`・`fit`・`predict` などのさまざまな機能を紹介するとともに、非線形な決定境界のために特に困難とされる問題である、2 つの交差する半円からなる合成問題への使用方法を示します。


## 関数の説明
この Qiskit Function を使用すると、Singularity の量子強化アンサンブル分類器を使って 2 値分類問題を解くことができます。内部では、ラベル付きデータセット上でアンサンブル分類器を古典的に学習するハイブリッドアプローチを採用し、その後 IBM&reg; QPU 上の量子近似最適化アルゴリズム（QAOA）を用いて最大の多様性と汎化性能が得られるよう最適化します。ユーザーフレンドリーなインターフェースを通じて、要件に合わせて分類器を設定し、任意のデータセットで学習させ、未知のデータセットに対して予測を行えます。

汎用的な分類問題を解くには、以下の手順を実施してください。
1. データセットを前処理し、学習セットとテストセットに分割します。必要に応じて、学習セットをさらに学習セットと検証セットに分割できます。これには [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html) を使用できます。
2. 学習セットがクラス不均衡の場合は、[imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets) を使用してクラスを均衡化するためにリサンプリングできます。
3. カタログの `file_upload` メソッドに都度関連するパスを渡して、学習・検証・テストセットをそれぞれ関数のストレージにアップロードします。
4. 関数の `create` アクションを使用して量子分類器を初期化します。このアクションは、学習器の数と種類・正則化（ラムダ値）・レイヤー数・古典オプティマイザーの種類・量子バックエンドなどの最適化オプションといったハイパーパラメータを受け付けます。
5. 関数の `fit` アクションを使用して、ラベル付き学習セット（および該当する場合は検証セット）を渡し、量子分類器を学習セットで学習します。
6. 関数の `predict` アクションを使用して、未知のテストセットに対して予測を行います。
---
## はじめに
[IBM Quantum Platform API キー](http://quantum.cloud.ibm.com/)を使用して認証し、以下のように Qiskit Function を選択してください。

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## 使用例
### データセットを分類する
この例では、「Singularity Machine Learning - Classification」関数を使用して、2 つの交差する三日月形の半円からなるデータセットを分類します。このデータセットは合成的かつ 2 次元であり、バイナリラベルが付与されています。重心ベースのクラスタリングや線形分類などのアルゴリズムに対して困難なデータセットとして作成されています。
![Moons データセット](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
このプロセスを通じて、分類器の作成・学習データへのフィット・テストデータへの予測・終了後の分類器の削除方法を学びます。
開始前に、[scikit-learn](https://scikit-learn.org/) をインストールしてください。以下のコマンドを使用してインストールできます。

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


以下の手順を実行してください。

1) [scikit-learn](https://scikit-learn.org/) の [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) 関数を使用して合成データセットを作成します。
2) 生成した合成データセットを[共有データディレクトリ](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html)にアップロードします。
3) [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create) アクションを使用して量子強化分類器を作成します。
4) [`list`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#list) アクションを使用して分類器の一覧を表示します。
5) [`fit`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#fit) アクションを使用して、学習データで分類器を学習させます。
6) [`predict`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#predict) アクションを使用して、学習済み分類器でテストデータを予測します。
7) [`delete`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#delete) アクションを使用して分類器を削除します。
8) 完了後にクリーンアップを行います。
**ステップ 1.** 必要なモジュールをインポートして合成データセットを生成し、学習データセットとテストデータセットに分割します。

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 3.** Create a quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


**Step 4.** Train the quantum-enhanced classifier using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


**ステップ 3.** [`create`](https://docs.quantum.ibm.com/api/functions/multiverse-computing-singularity#create) アクションを使用して量子強化分類器を作成します。

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


**Step 6.** Delete the quantum-enhanced classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


**Step 7.** Clean up local and shared data directories.

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

### `create_fit_predict` example

The following example demonstrates the [`create_fit_predict`](/docs/api/functions/multiverse-computing-singularity#create_fit_predict) action.

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Benchmarks

These benchmarks show that the classifier can achieve extremely high accuracies on challenging problems. They also show that increasing the number of learners in the ensemble (number of qubits) can lead to increased accuracy.

"Classical accuracy" refers to the accuracy obtained using corresponding classical state of the art which, in this case, is an AdaBoost classifier based on an ensemble of size 75. "Quantum accuracy", on the other hand, refers to the accuracy obtained using the "Singularity Machine Learning - Classification".

| Problem | Dataset Size | Ensemble Size | Number of Qubits | Classical Accuracy | Quantum Accuracy | Improvement |
|-------------|-------------|-------------|-------------|-------------|-------------|-------------|
| Grid stability | 5000 examples, 12 features | 55 | 55 |  76% | 91% | 15% |
| Grid stability | 5000 examples, 12 features | 65 | 65 |  76% | 92% | 16% |
| Grid stability | 5000 examples, 12 features | 75 | 75 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 85 | 85 |  76% | 94% | 18% |
| Grid stability | 5000 examples, 12 features | 100 | 100 |  76% | 95% | 19% |

----

As quantum hardware evolves and scales, the implications for our quantum classifier become increasingly significant. While the number of qubits does impose limitations on the size of the ensemble that can be utilized, it does not restrict the volume of data that can be processed. This powerful capability enables the classifier to efficiently handle datasets containing millions of data points and thousands of features. Importantly, the constraints related to ensemble size can be addressed through the implementation of a large-scale version of the classifier. By leveraging an iterative outer-loop approach, the ensemble can be dynamically expanded, enhancing flexibility and overall performance. However, it's worth noting that this feature has not yet been implemented in the current version of the classifier.

## Changelog

### 4 June 2025
- Upgraded `QuantumEnhancedEnsembleClassifier` with the following updates:
  - Added onsite/alpha regularization. You can specify `regularization_type` to be `onsite` or `alpha`
  - Added auto-regularization. You can set `regularization` to `auto` to use auto-regularization
  - Added `optimization_data` parameter to the `fit` method to choose optimization data for quantum optimization. You can use one of these options: `train`, `validation`, or `both`
  - Improved overall performance
- Added detailed status tracking for running jobs

### 20 May 2025
- Standardized error handling

### 18 March 2025
- Upgraded qiskit-serverless to 0.20.0 and base image to 0.20.1

### 14 February 2025
- Upgraded base image to 0.19.1

### 6 February 2025
- Upgraded qiskit-serverless to 0.19.0 and base image to 0.19.0

### 13 November 2024
- Release of Singularity Machine Learning - Classification

## Get support

For any questions, [reach out to Multiverse Computing](mailto:singularity@multiversecomputing.com).

Be sure to include the following information:

- The Qiskit Function Job ID (`job.job_id`)
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- Request access to [Multiverse Computing's Singularity Machine Learning Classification function](https://quantum.cloud.ibm.com/functions?id=multiverse-singularity).
- Visit the [API reference](/docs/api/functions/multiverse-computing-singularity) for this Qiskit Function.
- Review [Leclerc, L., et al. (2023). Financial risk management on a neutral atom quantum processor. Physical Review Research, 5, 043117](https://journals.aps.org/prresearch/pdf/10.1103/PhysRevResearch.5.043117).

</Admonition>